# Combined Cycle Power Plant - Regression

**Problem statement:** Predict net hourly electrical energy output (`PE`, MW) from ambient temperature (`AT`), exhaust vacuum (`V`), ambient pressure (`AP`), and relative humidity (`RH`).

**Source:** UCI Machine Learning Repository, dataset 294. The local CSV is generated by `data/download.py`.

## 1. Setup and reproducibility

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

RANDOM_STATE = 42
TARGET = "PE"
FEATURES = ["AT", "V", "AP", "RH"]

np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", palette="colorblind")

## 2. Dataset loading and audit

In [2]:
data_path = Path("data/regression.csv")
if not data_path.exists():
    data_path = Path("../data/regression.csv")

df = pd.read_csv(data_path)
assert list(df.columns) == FEATURES + [TARGET], "Unexpected dataset columns"

print(f"Shape: {df.shape}")
display(df.head())

Shape: (9568, 5)


,AT,V,AP,RH,PE
0,14.96,41.76,1024.07,73.17,463.26
1,25.18,62.96,1020.04,59.08,444.37
2,5.11,39.40,1012.16,92.14,488.56
3,20.86,57.32,1010.24,76.64,446.48
4,10.82,37.50,1009.23,96.62,473.90


In [3]:
audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean().mul(100).round(2),
    "unique_values": df.nunique(),
})
display(audit)
print(f"Duplicate rows: {df.duplicated().sum()}")

,dtype,missing_count,missing_percent,unique_values
AT,float64,0,0.0,2773
V,float64,0,0.0,634
AP,float64,0,0.0,2517
RH,float64,0,0.0,4546
PE,float64,0,0.0,4836


Duplicate rows: 41


In [4]:
display(df.describe().T)
display(df[TARGET].describe().to_frame("target_distribution"))

,count,mean,std,min,25%,50%,75%,max
AT,9568.0,19.651231,7.452473,1.81,13.5100,20.345,25.72,37.11
V,9568.0,54.305804,12.707893,25.36,41.7400,52.080,66.54,81.56
AP,9568.0,1013.259078,5.938784,992.89,1009.1000,1012.940,1017.26,1033.30
RH,9568.0,73.308978,14.600269,25.56,63.3275,74.975,84.83,100.16
PE,9568.0,454.365009,17.066995,420.26,439.7500,451.550,468.43,495.76


,target_distribution
count,9568.000000
mean,454.365009
std,17.066995
min,420.260000
25%,439.750000
50%,451.550000
75%,468.430000
max,495.760000


### Audit observation

The dataset contains 9,568 observations, four continuous predictors, and one continuous target. All columns load as `float64` and no values are missing, so imputation is unnecessary. There are 41 exact duplicate rows; these must be removed before splitting to prevent identical observations appearing in both train and test sets. `PE` spans 420.26-495.76 MW, with mean 454.37 MW and standard deviation 17.07 MW.